# Emotional Manipulation Detection - Demo
**CB011219 - Lasith Siriwardena**

In [ ]:
# 1. install dependencies
!pip install transformers torch --quiet

In [ ]:
# 2. upload and load the model
from google.colab import files
import zipfile, pickle, torch
import torch.nn.functional as F
from transformers import BertTokenizer, BertForSequenceClassification

uploaded = files.upload()

with zipfile.ZipFile('fyp_bert_model_v2.zip', 'r') as z:
    z.extractall('model')

device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model     = BertForSequenceClassification.from_pretrained('model').to(device)
tokenizer = BertTokenizer.from_pretrained('model')

with open('model/label_encoder.pkl', 'rb') as f:
    le = pickle.load(f)

model.eval()
print('ready')

In [ ]:
# 3. add a conversation to analyse
conversation = """
A: I cancelled my plans for you and you still wont come?
B: I told you I wasnt feeling well.
A: You never care about anyone but yourself. I always put you first.
B: Thats not fair.
A: If you actually cared about me you would make the effort.
"""

In [ ]:
# 4. run the analysis
import time

start    = time.time()
encoding = tokenizer(
    conversation,
    max_length=256,
    padding='max_length',
    truncation=True,
    return_tensors='pt'
)

ids  = encoding['input_ids'].to(device)
mask = encoding['attention_mask'].to(device)

with torch.no_grad():
    out   = model(input_ids=ids, attention_mask=mask)
    probs = F.softmax(out.logits, dim=1).squeeze().cpu().numpy()

end         = time.time()
predicted   = le.classes_[probs.argmax()]
neutral_idx = list(le.classes_).index('neutral')
manip_score = 1 - probs[neutral_idx]

print('=' * 50)
print(f'  Type:               {predicted}')
print(f'  Manipulation score: {manip_score:.1%}')
print('=' * 50)
print()
print('Breakdown:')
for cls, prob in sorted(zip(le.classes_, probs), key=lambda x: -x[1]):
    bar = chr(9608) * int(prob * 30)
    print(f'  {cls:<22} {prob:.1%}  {bar}')
print()
print(f'  Time: {(end - start) * 1000:.1f}ms')
print('=' * 50)